# Participant characteristics

Summarize demographic characteristics, study-arm allocation, AI use, and AI literacy.

## Load analysis data

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("..").resolve() / "data" / "Psy_Case_Vignette_Data_Prolific_analysis.xlsx"
COL = "Country of residence"

df = pd.read_excel(DATA_PATH)

if COL not in df.columns:
    raise KeyError(f"Column not found: {COL!r}. Available columns: {list(df.columns)}")

country = df[COL]
n_rows = len(country)
n_missing = country.isna().sum()
n_nonmissing = n_rows - n_missing
n_unique = country.nunique(dropna=True)

print(f"File: {DATA_PATH.name}")
print(f"Rows: {n_rows:,}")
print(f"{COL!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}, unique={n_unique:,}")
print()
print("Value counts:")
print(country.value_counts(dropna=False).to_string())
print()
print("Share (%):")
print((country.value_counts(normalize=True, dropna=False) * 100).round(2).astype(str) + "%")

## Summarize participant demographics

In [ ]:
CATEGORICAL_COLS = ["Sex", "Ethnicity simplified", "Employment status"]
NUMERIC_COLS = ["Age"]

missing_cols = [col for col in CATEGORICAL_COLS + NUMERIC_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")


def summarize_categorical(series: pd.Series) -> None:
    n_rows = len(series)
    n_missing = series.isna().sum()
    n_nonmissing = n_rows - n_missing
    n_unique = series.nunique(dropna=True)

    print(f"{series.name!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}, unique={n_unique:,}")
    print("Value counts:")
    print(series.value_counts(dropna=False).to_string())
    print("Share (%):")
    print((series.value_counts(normalize=True, dropna=False) * 100).round(2).astype(str) + "%")


for col in CATEGORICAL_COLS:
    print("=" * 60)
    summarize_categorical(df[col])
    print()

for col in NUMERIC_COLS:
    series = df[col]
    n_rows = len(series)
    n_missing = series.isna().sum()
    n_nonmissing = n_rows - n_missing

    print("=" * 60)
    print(f"{col!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}")
    print(series.describe().round(2).to_string())

## Summarize attention-check failures

In [ ]:
ATT_COLS = ["att_check_fail", "att_check2_fail"]
CATEGORICAL_COLS = ["Country of residence", "Sex", "Ethnicity simplified", "Employment status"]
NUMERIC_COLS = ["Age"]

missing_cols = [col for col in ATT_COLS + CATEGORICAL_COLS + NUMERIC_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

df_fail = df[(df["att_check_fail"] == 1) | (df["att_check2_fail"] == 1)].copy()

print(f"Attention check failures: {len(df_fail):,} of {len(df):,} ({len(df_fail) / len(df) * 100:.2f}%)")
print(f"  att_check_fail = 1 only: {((df_fail['att_check_fail'] == 1) & (df_fail['att_check2_fail'] == 0)).sum():,}")
print(f"  att_check2_fail = 1 only: {((df_fail['att_check_fail'] == 0) & (df_fail['att_check2_fail'] == 1)).sum():,}")
print(f"  both = 1: {((df_fail['att_check_fail'] == 1) & (df_fail['att_check2_fail'] == 1)).sum():,}")
print()

for col in CATEGORICAL_COLS:
    series = df_fail[col]
    n_rows = len(series)
    n_missing = series.isna().sum()
    n_nonmissing = n_rows - n_missing
    n_unique = series.nunique(dropna=True)

    print("=" * 60)
    print(f"{col!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}, unique={n_unique:,}")
    print("Value counts:")
    print(series.value_counts(dropna=False).to_string())
    print("Share (%):")
    print((series.value_counts(normalize=True, dropna=False) * 100).round(2).astype(str) + "%")
    print()

for col in NUMERIC_COLS:
    series = df_fail[col]
    n_rows = len(series)
    n_missing = series.isna().sum()
    n_nonmissing = n_rows - n_missing

    print("=" * 60)
    print(f"{col!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}")
    print(series.describe().round(2).to_string())

## Summarize study-arm allocation

In [ ]:
COL = "study_arm"

if COL not in df.columns:
    raise KeyError(f"Column not found: {COL!r}. Available columns: {list(df.columns)}")

study_arm = df[COL]
n_rows = len(study_arm)
n_missing = study_arm.isna().sum()
n_nonmissing = n_rows - n_missing
n_unique = study_arm.nunique(dropna=True)

print(f"{COL!r}: non-missing={n_nonmissing:,}, missing={n_missing:,}, unique={n_unique:,}")
print()
print("Value counts:")
print(study_arm.value_counts(dropna=False).to_string())
print()
print("Share (%):")
print((study_arm.value_counts(normalize=True, dropna=False) * 100).round(2).astype(str) + "%")

## Compare characteristics across study arms

In [ ]:
from scipy.stats import chi2_contingency, kruskal

ARM_LABELS = {
    1: "ChatGPT",
    2: "Copilot",
    3: "Perplexity",
    4: "Control",
}
ARM_ORDER = ["ChatGPT", "Copilot", "Perplexity", "Control"]
CATEGORICAL_COLS = [
    "Sex",
    "Country of residence",
    "Ethnicity simplified",
    "Employment status",
]

missing_cols = [col for col in ["study_arm", "Age", *CATEGORICAL_COLS] if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}. Available columns: {list(df.columns)}")

df_arm = df.copy()
df_arm["Arm"] = pd.Categorical(
    df_arm["study_arm"].map(ARM_LABELS),
    categories=ARM_ORDER,
    ordered=True,
)

print("Sample size by study arm")
print("=" * 60)
arm_n = df_arm.groupby("Arm", observed=True).size().rename("n")
arm_pct = (arm_n / len(df_arm) * 100).round(1).astype(str) + "%"
print(pd.concat([arm_n, arm_pct.rename("% of total")], axis=1).to_string())
print(f"\nTotal N = {len(df_arm):,}")
print()


def crosstab_by_arm(data: pd.DataFrame, col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    counts = pd.crosstab(data[col], data["Arm"], dropna=False)
    counts = counts.reindex(columns=ARM_ORDER, fill_value=0)
    pct_within_arm = counts.div(counts.sum(axis=0), axis=1) * 100
    return counts, pct_within_arm.round(1)


for col in CATEGORICAL_COLS:
    counts, pct = crosstab_by_arm(df_arm, col)
    print("=" * 60)
    print(f"{col} by study arm")
    print("\nCounts:")
    print(counts.to_string())
    print("\nRow share within each arm (%):")
    print(pct.astype(str) + "%")

    table = pd.crosstab(df_arm[col], df_arm["Arm"], dropna=False)
    if table.shape[0] > 1 and table.shape[1] > 1:
        chi2, p, _, _ = chi2_contingency(table.values)
        print(f"\nChi-square test across arms: chi2 = {chi2:.2f}, p = {p:.4f}")
    print()

print("=" * 60)
print("Age by study arm")
age_by_arm = (
    df_arm.groupby("Arm", observed=True)["Age"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .round(2)
)
age_by_arm.columns = ["n", "mean", "sd", "median", "min", "max"]
print(age_by_arm.to_string())

age_groups = [group["Age"].dropna().to_numpy() for _, group in df_arm.groupby("Arm", observed=True)]
if all(len(g) > 0 for g in age_groups):
    h_stat, p_age = kruskal(*age_groups)
    print(f"\nKruskal-Wallis test across arms: H = {h_stat:.2f}, p = {p_age:.4f}")

## Summarize AI chatbot use

In [ ]:
from scipy.stats import chi2_contingency, kruskal

AI_FREQ_LABELS = {
    1: "Daily",
    2: "Several times a week",
    3: "Once a week",
    4: "A few times a month",
    5: "Rarely",
    6: "I have stopped using them",
}

if "ai_frequency" not in df.columns:
    raise KeyError(f"Column not found: 'ai_frequency'. Available columns: {list(df.columns)}")

freq = df["ai_frequency"].dropna().astype(int)
print("AI chatbot use frequency — whole sample")
print("=" * 60)
print(f"Non-missing: {freq.shape[0]:,} of {len(df):,} ({freq.shape[0] / len(df) * 100:.1f}%)")
print(f"Missing: {df['ai_frequency'].isna().sum():,}")
print()

labeled = freq.map(AI_FREQ_LABELS)
print("Value counts:")
print(labeled.value_counts().to_string())
print()
print("Share (%):")
print((labeled.value_counts(normalize=True) * 100).round(2).astype(str) + "%")
print()

df_freq = df.dropna(subset=["ai_frequency"]).copy()
df_freq["ai_frequency"] = df_freq["ai_frequency"].astype(int)
df_freq["ai_frequency_label"] = df_freq["ai_frequency"].map(AI_FREQ_LABELS)
df_freq["Arm"] = pd.Categorical(
    df_freq["study_arm"].map(ARM_LABELS),
    categories=ARM_ORDER,
    ordered=True,
)

counts = pd.crosstab(df_freq["ai_frequency_label"], df_freq["Arm"])
counts = counts.reindex(
    [AI_FREQ_LABELS[i] for i in sorted(AI_FREQ_LABELS)],
    fill_value=0,
)
counts = counts.reindex(columns=ARM_ORDER, fill_value=0)

print("AI chatbot use frequency by study arm")
print("=" * 60)
print("\nCounts:")
print(counts.to_string())
print("\nRow share within each arm (%):")
print((counts.div(counts.sum(axis=0), axis=1) * 100).round(1).astype(str) + "%")

table = pd.crosstab(df_freq["ai_frequency"], df_freq["Arm"])
chi2, p_chi, _, _ = chi2_contingency(table.values)
print(f"\nChi-square test across arms: chi2 = {chi2:.2f}, p = {p_chi:.4f}")

freq_groups = [
    group["ai_frequency"].to_numpy()
    for _, group in df_freq.groupby("Arm", observed=True)
]
h_stat, p_kw = kruskal(*freq_groups)
print(f"Kruskal-Wallis test across arms: H = {h_stat:.2f}, p = {p_kw:.4f}")

print("\nMedian by arm:")
print(
    df_freq.groupby("Arm", observed=True)["ai_frequency"]
    .agg(["count", "median"])
    .reindex(ARM_ORDER)
    .to_string()
)

## Calculate and summarize AI literacy

In [ ]:
AILI_ITEMS = [f"aili_{i:02d}" for i in range(1, 13)]
REVERSE_ITEMS = {"aili_02", "aili_05", "aili_11"}

missing_cols = [col for col in AILI_ITEMS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}")


def recode_aili(series: pd.Series, reverse: bool = False) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    if reverse:
        values = 8 - values
    return values


aili_recoded = pd.DataFrame(
    {
        col: recode_aili(df[col], reverse=col in REVERSE_ITEMS)
        for col in AILI_ITEMS
    }
)

df_aili = df.copy()
df_aili["ai_literacy_sum"] = aili_recoded.sum(axis=1, min_count=len(AILI_ITEMS))

complete = df_aili["ai_literacy_sum"].notna()
print("AI literacy sum score — whole sample")
print("=" * 60)
print(f"Complete cases (all 12 items): {complete.sum():,} of {len(df):,}")
print(f"Missing/incomplete: {(~complete).sum():,}")
print()
print(df_aili.loc[complete, "ai_literacy_sum"].describe().round(2).to_string())
print()
print("Reverse-coded items: aili_02, aili_05, aili_11")
print("Possible range after recoding: 12–84")
print()

df_aili["Arm"] = pd.Categorical(
    df_aili["study_arm"].map(ARM_LABELS),
    categories=ARM_ORDER,
    ordered=True,
)

print("AI literacy sum score by study arm")
print("=" * 60)
by_arm = (
    df_aili.dropna(subset=["ai_literacy_sum"])
    .groupby("Arm", observed=True)["ai_literacy_sum"]
    .agg(["count", "mean", "std", "median", "min", "max"])
    .reindex(ARM_ORDER)
    .round(2)
)
print(by_arm.to_string())

score_groups = [
    group["ai_literacy_sum"].dropna().to_numpy()
    for _, group in df_aili.groupby("Arm", observed=True)
]
h_stat, p_kw = kruskal(*score_groups)
print(f"\nKruskal-Wallis test across arms: H = {h_stat:.2f}, p = {p_kw:.4f}")

## Control arm: self-reported resources used

Free-text responses in `vig_*_post_control_resource` (control arm only). Responses are split on list delimiters (preserving URLs), mapped to canonical resource labels via domain/alias rules, then summarized as **top 5** at participant level (used ≥1× across any vignette) and response level (each vignette response; multi-label allowed).

In [ ]:
import re
from collections import Counter

VIGNETTES = list("ABCDE")
RESOURCE_COLS = [f"vig_{v}_post_control_resource" for v in VIGNETTES]
TOP_N = 5

missing_cols = [col for col in RESOURCE_COLS if col not in df.columns]
if missing_cols:
    raise KeyError(f"Columns not found: {missing_cols}")

URL_TOKEN = re.compile(r"https?://\S+|www\.\S+")
SPLIT_RE = re.compile(r"\s*[,;]|\s+\band\b|\s+\bor\b|\s*\+\s*", re.I)

CANONICAL_RULES = [
    ("MedlinePlus", [r"medline\s*plus", r"medlineplus", r"mediline", r"nlm\.nih"]),
    ("NHS", [r"\bnhs\b", r"nhs\.uk", r"nhsinform", r"nhs24"]),
    ("NIH / NIMH", [r"\bnimh\b", r"\bnih\b"]),
    ("PubMed", [r"pubmed"]),
    ("WebMD", [r"webmd"]),
    ("Mayo Clinic", [r"mayo\s*clinic", r"mayoclinic"]),
    ("Cleveland Clinic", [r"cleveland\s*clinic", r"my\.clevelandclinic"]),
    ("Healthline", [r"healthline"]),
    ("Mind UK", [r"mind\.org"]),
    ("Google", [r"\bgoogle\b", r"google chrome", r"web search", r"i googled"]),
    ("ChatGPT / AI", [r"chat\s*gpt", r"chatgpt"]),
    ("Ubie", [r"\bubie\b"]),
]

MANUAL_ALIASES = {
    "medline": "MedlinePlus",
    "medicineplus": "MedlinePlus",
    "medlineplue": "MedlinePlus",
}

DOMAIN_RULES = {
    "nhs": "NHS",
    "medlineplus": "MedlinePlus",
    "nlm.nih": "MedlinePlus",
    "webmd": "WebMD",
    "mayoclinic": "Mayo Clinic",
    "clevelandclinic": "Cleveland Clinic",
    "healthline": "Healthline",
    "mind.org": "Mind UK",
}


def normalize_empty(val):
    if pd.isna(val):
        return None
    text = str(val).strip()
    if text == "" or text.lower() == "nan":
        return None
    return text


def tokenize(text: str) -> list[str]:
    placeholders: list[str] = []

    def repl(match):
        placeholders.append(match.group(0).rstrip(".,)"))
        return f"__URL{len(placeholders) - 1}__"

    masked = URL_TOKEN.sub(repl, text)
    parts = [part.strip() for part in SPLIT_RE.split(masked) if part.strip()]
    fragments = []
    for part in parts:
        for i, url in enumerate(placeholders):
            part = part.replace(f"__URL{i}__", url)
        part = part.strip()
        if part:
            fragments.append(part)
    return fragments


def canonicalize(fragment: str) -> str | None:
    text = fragment.strip()
    if not text:
        return None

    lower = text.lower()
    if lower in MANUAL_ALIASES:
        return MANUAL_ALIASES[lower]

    for label, patterns in CANONICAL_RULES:
        if any(re.search(pattern, lower, re.I) for pattern in patterns):
            return label

    domain_match = re.search(r"([a-z0-9-]+(?:\.[a-z0-9-]+)+)", lower, re.I)
    if domain_match:
        domain = domain_match.group(1).lower()
        for key, label in DOMAIN_RULES.items():
            if key in domain:
                return label

    if len(text) > 90:
        return None

    return None


def labels_for_response(raw: str) -> set[str]:
    return {
        label
        for fragment in tokenize(raw)
        for label in [canonicalize(fragment)]
        if label is not None
    }


df_control = df.loc[df["study_arm"] == 4].copy()
n_participants = len(df_control)

response_counts = Counter()
n_responses = 0
for col in RESOURCE_COLS:
    for raw in df_control[col].map(normalize_empty).dropna():
        n_responses += 1
        for label in labels_for_response(raw):
            response_counts[label] += 1

participant_counts = Counter()
for _, row in df_control.iterrows():
    labels = set()
    for col in RESOURCE_COLS:
        raw = normalize_empty(row[col])
        if raw:
            labels.update(labels_for_response(raw))
    for label in labels:
        participant_counts[label] += 1


def print_top_n(title: str, counts: Counter, denominator: int) -> None:
    print(title)
    print("=" * 60)
    rows = []
    for rank, (label, n) in enumerate(counts.most_common(TOP_N), start=1):
        pct = 100 * n / denominator
        rows.append({"rank": rank, "resource": label, "n": n, "pct": f"{pct:.1f}%"})
    print(pd.DataFrame(rows).to_string(index=False))
    print()


print(f"Control arm participants: {n_participants:,}")
print(f"Vignette responses: {n_responses:,} ({n_participants:,} x {len(VIGNETTES)})")
print("Standardization: split on commas/and/or (URLs kept intact); map via CANONICAL_RULES, MANUAL_ALIASES, DOMAIN_RULES")
print()

print_top_n(
    f"Top {TOP_N} — participant level (used >=1x across any vignette)",
    participant_counts,
    n_participants,
)
print_top_n(
    f"Top {TOP_N} — response level (each vignette response; multi-label)",
    response_counts,
    n_responses,
)